Statistical Analysis

Goal: Use statistical tests to evaluate whether observed differences are supported by evidence.

Define a hypothesis

I will investigate this question:

Do average daily sales differ between weekends and weekdays?

Define the hypotheses:

Null hypothesis, H₀: Mean daily sales are equal on weekdays and weekends.
Alternative hypothesis, H₁: Mean daily sales differ between weekdays and weekends.

Run a Welch's t-test

In [1]:
!pip install scipy

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.1.2 -> 26.2.2
[notice] To update, run: C:\Users\Admin\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [6]:
import pandas as pd
import numpy as np

from scipy import stats

daily = pd.read_csv(
    "data/processed/Daily_Sales.csv",
    parse_dates=["Invoice_Date"]
)

daily["Is_Weekend"] = (
    daily["Invoice_Date"].dt.dayofweek >= 5
)

weekend_sales = daily.loc[
    daily["Is_Weekend"],
    "Daily_Revenue"
].dropna()

weekday_sales = daily.loc[
    ~daily["Is_Weekend"],
    "Daily_Revenue"
].dropna()

t_stat, p_value = stats.ttest_ind(
    weekend_sales,
    weekday_sales,
    equal_var=False
)

difference = (
    weekend_sales.mean()
    - weekday_sales.mean()
)

print("Average weekday sales:", weekday_sales.mean())

print("Average weekend sales:", weekend_sales.mean())

print("Weekend minus weekday:", difference)

print("T-statistic:", t_stat)

print("P-value:", p_value)

Average weekday sales: 159768.62191570885
Average weekend sales: 200739.39682692304
Weekend minus weekday: 40970.77491121419
T-statistic: 11.267940871701235
P-value: 1.5155706340118392e-22


Calculate a 95% confidence interval

In [7]:
n1 = len(weekend_sales)
n0 = len(weekday_sales)

v1 = weekend_sales.var(ddof=1)
v0 = weekday_sales.var(ddof=1)

se = np.sqrt(
    v1 / n1 + v0 / n0
)

df = (
    (v1 / n1 + v0 / n0) ** 2
    /
    (
        (v1 / n1) ** 2 / (n1 - 1)
        +
        (v0 / n0) ** 2 / (n0 - 1)
    )
)

critical_value = stats.t.ppf(
    0.975,
    df
)

lower = difference - critical_value * se
upper = difference + critical_value * se

print("95% confidence interval:")
print(lower, upper)

95% confidence interval:
33795.1749754974 48146.37484693098


Interpret the results

| Result                                  | Interpretation                                         |
| --------------------------------------- | ------------------------------------------------------ |
| P-value below 0.05                      | Evidence against equal mean daily sales                |
| P-value at least 0.05                   | Insufficient evidence to reject equal mean daily sales |
| Confidence interval entirely above zero | Estimated weekend mean is higher                       |
| Confidence interval entirely below zero | Estimated weekend mean is lower                        |
| Confidence interval includes zero       | The interval does not rule out no difference           |
